# Pipeline

## Adding New Files

The first step is to scan for new video files in the watch directory (/home/amos/media/tv/). 

In [5]:
def add_to_queue(d,
                 engine,
                 extensions=('.mp4', '.m4v', '.mkv', '.avi')):
    """
    Scan directory looking for new files. Then, match file to IMDb ID to get metadata.
    Finally, add the new files with their corresponding IMDb IDs to the SQL server. 
    """
    paths = [x for x in get_files(d, extensions=extensions) if 'sample' not in x.stem]
    df = pd.DataFrame(parse_paths(paths))
    logging.debug(f'Found {df.shape[0]} files.')

    with engine.connect() as conn:
        existing = pd.read_sql_query('SELECT filename FROM queue;', conn)
        new = df[~df['filename'].isin(existing['filename'])]
        new = new[(new['title'].notna()) &
                  (new['season'].notna()) &
                  (new['episode'].notna())]
        logging.info(f'{new.shape[0]} new files out of {df.shape[0]}.')
        new.to_sql('queue', conn, if_exists='append', index=False)
        conn.commit()

If a given file is not already in the database, then it uses regex to extract the title (and year) from the filename.

In [6]:
def get_id(title,
           year,
           kind=None,
           log_level=20):
    import re
    from imdb import Cinemagoer

    special_char_map = {ord('ä'): 'a',
                        ord('ü'): 'u',
                        ord('ö'): 'o', 
                        ord('ß'): 's', 
                        ord('ō'): 'o'}
    ia = Cinemagoer(loggingLevel=log_level)

    result = ia.search_movie(title)
    temp = [x for x in result if kind in x.data['kind']] if kind else result
    results = []
    for x in temp:
        a = re.sub('[^0-9a-zA-Z]+', '', x.data['title'].translate(special_char_map)).lower()
        b = re.sub('[^0-9a-zA-Z]+', '', title).lower()
        if a == b:
            results.append(x)
    # results = [x for x in temp if re.sub('[^0-9a-zA-Z]+', '', title) ==
    #            re.sub('[^0-9a-zA-Z]+', '', x.data['title'])]
    if not results:
        results = match_title(temp, title, char_map=special_char_map, threshold=0.0)

    year = int(year)
    for r in results:
        try:
            imdb_year = int(r.data['year'])
            if year - 2 <= imdb_year <= year + 2:
                return r.movieID
        except KeyError:
            continue
    return None


Next, we have to pull info from IMDb to decide whether or not a given file should undergo analysis. For instance, I exclude animation because it cannot be used for facial recognition.

In [7]:
def update_queue(engine):
    with engine.connect() as conn:
        df = pd.read_sql_query('SELECT * FROM queue WHERE series_id IS NULL', conn)
        g = df.groupby('title').max().reset_index()
        logging.debug(f'Found {g.shape[0]} to process.')
        for idx, row in tqdm(g.iterrows(),
                             desc='Adding to queue ...',
                             total=g.shape[0],
                             leave=False):
            imdb_id = imdb_id_from_row(row)
            if not imdb_id:
                continue 
            row['series_id'] = int(imdb_id) 
            row = get_metadata(row)
            conn.execute(db.text(f"""
                                 UPDATE queue 
                                 SET series_id = {str(imdb_id)},
                                     year = {row["year"]},
                                     processed = 1
                                 WHERE title = '{row["title"].replace("'", "''")}'
                                 """))
            conn.commit()

## Analysis

First, connect to database with the correct credentials. 

In [8]:
def process_queue(engine, 
                  dst,
                  repo_name='CineFace',
                  token_path='./data/pat.txt',
                  branch='main',
                  series_id=None):
    with engine.connect() as conn:
        if series_id:
            queue = pd.read_sql_query(f"""
                                  SELECT *
                                  FROM queue
                                  WHERE to_analyze = 1 AND 
                                        analyzed = 0 AND 
                                        series_id = {series_id}
                                  ORDER BY height ASC 
                                  """, conn)
        else:
            queue = pd.read_sql_query("""
                                    SELECT *
                                    FROM queue
                                    WHERE to_analyze = 1 AND 
                                            analyzed = 0 AND 
                                            series_id IS NOT NULL
                                    ORDER BY height ASC 
                                    """, conn)
        
        logging.debug(f'Found {queue.shape[0]} for analysis.')
        for _, row in tqdm(queue.iterrows(), total=queue.shape[0]):
            try:
                a = Analyzer(row, dst, conn).analyze()
            except KeyError:
                conn.execute(db.text(f"""
                                    UPDATE queue
                                    SET to_analyze = -1
                                    WHERE series_id = {row["series_id"]} AND
                                        season = {row["season"]} AND
                                        episode = {row["episode"]}
                                    """))
                conn.commit()
                continue 
            if a.success:
                conn.execute(db.text(f"""
                                    UPDATE queue
                                    SET analyzed = 1, to_analyze = 0
                                    WHERE series_id = {row["series_id"]} AND
                                        season = {row["season"]} AND
                                        episode = {row["episode"]}
                                    """))
                conn.commit()
                logging.debug(f'Analyzed {row["title"]} ({row["series_id"]}) and saved results to {str(a.fp)}.')

## Add to Server

The final step is to add the new data to the server. First, we add the metadata for the series to the MySQL server.

In [10]:
def add_series_to_server(series_id, conn):
    tv = TV()
    imdb_id = f'tt{str(series_id).zfill(7)}'    # The imdb_id is stored as an integer in the database. Convert to formatted string.
    search = Find()
    results = search.find_by_imdb_id(imdb_id)
    tmdb_id = results['tv_results'][0]['id']    # Data is stored by imdb_id but IMDb API lacks episode info. Get from TMDb instead.
    results = tv.details(tmdb_id)
    datum = format_tmdb_series(results)
    datum['imdb_id'] = series_id
    df = pd.DataFrame([datum])
    try:
        df.to_sql('series',
                  conn, 
                  if_exists='append', 
                  index=False)
        conn.commit()
    except db.exc.IntegrityError:               # Skips if entry already in database.
        pass

The reason why I'm using both IMDb and TMDb is that I originally was using IMDb, but the API functionality pertaining to retrieving episodes from a series broke, so I had switch to TMDb. However, I used the IMDb extensively enough that I can't fully replace all references to it. This system is working for now, but I'd like to eventually settle on using either, not both. 

Next, we need to add each .csv file containing the face data from our analysis to the MySQL server.  

In [11]:
def add_file_to_server(file, conn):
    search = Find()
    episode = Episode()
    
    df = pd.read_csv(str(file), index_col=0)
    series_id = df.at[0, 'series_id']
    season_num = df.at[0, 'season']
    episode_num = df.at[0, 'episode']
    
    # Getting episode info from IMDb is not currently working, so we have to get the data from TMDb (using imdb_id).
    imdb_id = f'tt{str(series_id).zfill(7)}'    # I store the ids as integers in the database, but TMDb wants the 7-character string version.
    results = search.find_by_imdb_id(imdb_id)
    id_ = results['tv_results'][0]['id']
    
    # Before getting the episode info, check to see if the episode is already in the database.
    episode_df = pd.read_sql_query(f"""
                                     SELECT *
                                     FROM episodes
                                     WHERE series_id = {id_} AND
                                           season = {season_num} AND
                                           episode = {episode_num}
                                     """, conn)
    if episode_df.shape[0] == 0:
        try:
            e = episode.details(id_, season_num, episode_num)
            c = episode.credits(id_, season_num, episode_num)
            datum = {'series_id': id_,
                'imdb_id': series_id,
                'episode_id': e['id'],
                'title': e['name'],
                'season': season_num,
                'episode': episode_num,
                'air_date': e['air_date'],
                'cast': ','.join([str(x['id']) for x in c['cast']])}
            temp = pd.DataFrame([datum])
            try:
                temp.to_sql('episodes', conn, if_exists='append', index=False)
                conn.commit()      
            except db.exc.IntegrityError:
                pass 
            df = df.assign(episode_id=e['id'])
        except exceptions.TMDbException:        # Some episodes are missing from TMDb, unfortunately. Skips if missing.
            pass
    else:
        df = df.assign(episode_id=episode_df.at[0, 'episode_id'])
    
    
    # I generate a uuid to connect the face data in the SQL database to the encoding in the vector database.    
    df = df.assign(encoding_id=df['encoding'].map(lambda x: str(uuid.uuid4()) if not pd.isnull(x) else np.nan))
    temp = df.drop(['filepath', 'encoding'], axis=1)
    
    try:   
        temp.to_sql('faces', conn, if_exists='append', index=False)
        conn.commit()
    except db.exc.IntegrityError:       # Skips if the entry is already in the database.
        pass
    
    df = df[df['encoding'].notna()]

    # I store the encodings in a separate vector database to facilitate searching/matching.
    inject_encodings(df)
       

To facilitate facial recognition, facial embeddings are stored in a vector database. I use Qdrant for this. Connect to the Qdrant client.

In [13]:
from qdrant_client import QdrantClient, http

CLIENT = QdrantClient(host='192.168.0.131', port=6333) 

Next, check is the "FacialEmbeddings" database is present. Create if not.

In [14]:
collections = [x.name for x in CLIENT.get_collections().collections]
if 'FacialEmbeddings' not in collections:
    CLIENT.recreate_collection(collection_name='FacialEmbeddings',
                                vectors_config=VectorParams(size=128, distance=Distance.COSINE))

## Run Stored Procedures

Once all the data is safely in the database, we can run some further calcuations. Three measures should be calculated: pct_of_frame, num_faces, distance_from_center.

In [17]:
%%sql

CREATE DEFINER=`amos`@`%` PROCEDURE `getFaces`()
BEGIN
    UPDATE series
    LEFT JOIN (
        SELECT
            faces.series_id AS series_id,
            ROUND(AVG(faces.pct_of_frame), 4) as pct_of_frame
        FROM faces
        GROUP BY faces.series_id
    ) a
    ON series.tmdb_id = a.series_id
    LEFT JOIN (
        SELECT
            faces.series_id AS series_id,
            ROUND(AVG(faces.face_num) + 1, 3) AS num_faces
        FROM faces
        GROUP BY faces.series_id, faces.frame_num
    ) b
    ON series.tmdb_id = b.series_id
    SET series.avg_face_pct_of_frame = a.pct_of_frame,
        series.avg_faces_per_frame = b.num_faces;

    UPDATE episodes
    LEFT JOIN (
        SELECT
            faces.episode_id AS episode_id,
            ROUND(AVG(faces.pct_of_frame), 4) as pct_of_frame
        FROM faces
        GROUP BY faces.episode_id
    ) a
    ON episodes.episode_id = a.episode_id
    LEFT JOIN (
        SELECT
            bb.episode_id,
            ROUND(AVG(bb.num_faces), 3) as num_faces
        FROM (
            SELECT
                faces.episode_id AS episode_id,
                ROUND(AVG(faces.face_num) + 1, 3) AS num_faces
            FROM faces
            GROUP BY faces.episode_id, faces.frame_num
             ) bb
        GROUP BY bb.episode_id
        ) b
    ON episodes.episode_id = b.episode_id
    SET episodes.avg_face_pct_of_frame = a.pct_of_frame,
        episodes.avg_faces_per_frame = b.num_faces;
END

UsageError: Cell magic `%%sql` not found.
